In [0]:
# 1. IMPORT LIBRARIES
# ==============================
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:

%run /Workspace/Users/pothuritarun11@gmail.com/Data-Engineering-FMCG-Project/1_setup/utilities

In [0]:
# 3. WIDGETS
# ==============================
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
base_path = f's3://sportbar-rahul/products.csv'
print(base_path)


In [0]:
# 4. BRONZE LAYER
# ==============================
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)


In [0]:
df.printSchema()
display(df.limit(10))

In [0]:
# WRITE TO BRONZE
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")




In [0]:
# 5. READ FROM BRONZE
# ==============================
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
# 6. SILVER TRANSFORMATIONS
# ==============================

# 6.1 DROP DUPLICATES
print("Rows before:", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["product_id"])
print("Rows after:", df_silver.count())


In [0]:
#6.2 TITLE CASE FIX
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
     .otherwise(F.initcap("category"))
)


In [0]:
# 6.3 FIX SPELLING (Protien → Protein)
df_silver = (
    df_silver
    .withColumn(
        "product_name",
        F.regexp_replace("product_name", "(?i)Protien", "Protein")
    )
    .withColumn(
        "category",
        F.regexp_replace("category", "(?i)Protien", "Protein")
    )
)

In [0]:
# 7. STANDARDIZATION
# ==============================

# 7.1 ADD DIVISION COLUMN
df_silver = df_silver.withColumn(
    "division",
    F.when(F.col("category") == "Energy Bars", "Nutrition Bars")
     .when(F.col("category") == "Protein Bars", "Nutrition Bars")
     .when(F.col("category") == "Granola & Cereals", "Breakfast Foods")
     .when(F.col("category") == "Recovery Dairy", "Dairy & Recovery")
     .when(F.col("category") == "Healthy Snacks", "Healthy Snacks")
     .when(F.col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
     .otherwise("Other")
)

In [0]:
# 7.2 EXTRACT VARIANT (e.g., 60g)
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract("product_name", r"\((.*?)\)", 1)
)


In [0]:
# 7.3 CREATE PRODUCT CODE + CLEAN ID
df_silver = (
    df_silver
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit("999999"))
    )
    .withColumnRenamed("product_name", "product")
)

In [0]:
# 7.4 FINAL COLUMN ORDER
df_silver = df_silver.select(
    "product_code",
    "division",
    "category",
    "product",
    "variant",
    "product_id",
    "read_timestamp",
    "file_name",
    "file_size"
)

display(df_silver.limit(10))

In [0]:
# 8. WRITE TO SILVER
# ==============================
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

df_gold = df_silver.select(
    "product_code",
    "product_id",
    "division",
    "category",
    "product",
    "variant"
)

df_gold.show(5)

In [0]:
df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_products")

df_child_products = spark.sql(f"""
    SELECT product_code, division, category, product, variant 
    FROM fmcg.gold.sb_dim_products
""")

delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()